# Step 03 — Tile Boundary Grid Generator

This notebook generates a grid of Web Mercator tile boundary polygons at a specific zoom level for an Area of Interest (AOI). The output can be either a GeoJSON or a CSV file (with WKT geometries), and includes interactive leafmap previews of randomly selected tiles from the grid.

## 1. Install Necessary Libraries

We will use the following libraries:
- [mercantile](https://github.com/mapbox/mercantile): For Spherical Mercator tile calculations.
- [geopandas](https://geopandas.org/): For geographic data manipulation.
- [shapely](https://shapely.readthedocs.io/): For geometric operations.
- [fiona](https://fiona.readthedocs.io/): For file format support.

In [ ]:
!pip install -q --upgrade mercantile geopandas shapely fiona

## 2. Declare Variables

Input parameters for the grid generation process. Define the zoom level, output format, AOI file path, and the output name.

In [ ]:
# The zoom level for the tile grid
zoom = 17

# Output format: 'geojson' or 'csv'
output_format = 'geojson'

# Path to the Area of Interest (AOI) GeoJSON file
aoi_path = '../data/CZU_Fire_Perimeter_-2212360052269988636.geojson'

# Name for the output file
output_name = 'Z17_CZU_tile_boundary_grid'

## 3. Load the AOI and Generate Candidate Tiles

Load the AOI, ensure it is in WGS84, and use its bounding box to generate an initial set of candidate tiles with `mercantile.tiles()`. We also dissolve all AOI features into a single unified geometry for the intersection test in the next step.

In [ ]:
import mercantile
import geopandas as gpd
from shapely.geometry import shape, box, mapping
import pandas as pd

# Load the AOI
aoi_gdf = gpd.read_file(aoi_path)

# Ensure AOI is in WGS84 (EPSG:4326)
if aoi_gdf.crs != 'EPSG:4326':
    aoi_gdf = aoi_gdf.to_crs('EPSG:4326')

# Combine all features in the AOI for bounds calculation
bounds = aoi_gdf.total_bounds # [west, south, east, north]

# Get all tiles within the bounding box of the AOI
tiles = list(mercantile.tiles(*bounds, zoom))

print(f"Generated {len(tiles)} tiles at zoom level {zoom}")

## 4. Filter Tiles by Intersection & Build Tile Data

Rather than keeping every tile in the bounding box, we test each candidate tile's polygon against the dissolved AOI geometry using Shapely's `intersects()` method. Only tiles that actually overlap the input features are retained. This is especially useful when the AOI is an irregular polygon, a line, or a set of points — it avoids including tiles that fall in the corners of the bounding box but outside the actual features.

In [ ]:
from shapely.ops import unary_union

# Dissolve all AOI features into a single geometry for intersection testing
aoi_union = unary_union(aoi_gdf.geometry)

# Initialize the list to hold tile data
tile_data = []

# Track how many candidates came from the bounding box
num_candidates = len(tiles)

for tile in tiles:
    # Get the bounding box of the tile
    t_bounds = mercantile.bounds(tile)  # LngLatBbox(west, south, east, north)
    t_geom = box(t_bounds.west, t_bounds.south, t_bounds.east, t_bounds.north)

    # Skip tiles that do not intersect the actual AOI geometry
    if not aoi_union.intersects(t_geom):
        continue

    # Calculate properties
    zxy_val = f"{tile.z},{tile.x},{tile.y}"
    url_suffix = f"{tile.z}/{tile.x}/{tile.y}"

    # Upper left (west, north) and Lower right (east, south)
    upper_left = f"{t_bounds.west},{t_bounds.north}"
    lower_right = f"{t_bounds.east},{t_bounds.south}"

    # Centroid
    centroid = t_geom.centroid
    centroid_val = f"{centroid.x},{centroid.y}"

    tile_data.append({
        'zxy_val': zxy_val,
        'z_val': int(tile.z),
        'x_val': int(tile.x),
        'y_val': int(tile.y),
        'url_suffix': url_suffix,
        'upper_left': upper_left,
        'lower_right': lower_right,
        'centroid': centroid_val,
        'geometry': t_geom
    })

# Create a GeoDataFrame from only the intersecting tiles
grid_gdf = gpd.GeoDataFrame(tile_data, crs='EPSG:4326')

print(f"Retained {len(grid_gdf)} tiles that intersect the AOI (from {num_candidates} candidates)")

## 5. Exporting Results

Depending on the `output_format` variable, save the boundaries to GeoJSON or CSV. For CSV, convert the geometry to WKT (Well-Known Text) for compatibility.

In [ ]:
output_path = f"../output/{output_name}.{output_format if output_format == 'geojson' else 'csv'}"

if output_format == 'geojson':
    grid_gdf.to_file(output_path, driver='GeoJSON')
    print(f"Exported to GeoJSON: {output_path}")
elif output_format == 'csv':
    # Convert geometry to WKT for CSV/BigQuery ingest
    grid_gdf['geometry'] = grid_gdf['geometry'].apply(lambda x: x.wkt)
    grid_gdf.to_csv(output_path, index=False)
    print(f"Exported to CSV with WKT: {output_path}")

## 6. Install leafmap

[leafmap](https://leafmap.org/) is an open-source Python package for interactive geospatial visualization built on top of [ipyleaflet](https://ipyleaflet.readthedocs.io/) and [folium](https://python-visualization.github.io/folium/). We'll use it to preview a randomly selected tile from the grid, overlaid with a randomly chosen tile service from `output/tile_urls.csv`.

In [ ]:
!pip install -q --upgrade leafmap

## 7. Pick a Random Tile Service and Random Grid Tile

Load the tile service URLs from `output/tile_urls.csv`, then use Python's `random` module to select one service and one tile from `grid_gdf`. We extract the tile's centroid and bounds so we can center and zoom the map correctly.

In [ ]:
import random
import re
import pandas as pd

# Load tile service URLs
tile_services_df = pd.read_csv('../output/tile_urls.csv')

# Randomly select one tile service
random_service = tile_services_df.sample(1).iloc[0]
service_name = random_service['name']
service_url = random_service['tile_url']

# Parse the NAIP year and band type (RGB vs IR) from the service name
# Expected format: gee_naip_[ir_]YYYY
year_match = re.search(r'(\d{4})', service_name)
naip_year = year_match.group(1) if year_match else 'Unknown'
band_type = 'IR (Near-Infrared)' if '_ir_' in service_name else 'RGB (Natural Color)'

# Randomly select one tile from the grid
random_tile = grid_gdf.sample(1).iloc[0]

# Extract the centroid coordinates (lat, lon) for map centering
cx, cy = random_tile.geometry.centroid.x, random_tile.geometry.centroid.y

print(f"NAIP Year:    {naip_year}")
print(f"Band Type:    {band_type}")
print(f"Tile service: {service_name}")
print(f"Random tile:  {random_tile['zxy_val']}")
print(f"Centroid:     {cy:.6f}, {cx:.6f}")

## 8. Display the Tile on an Interactive Map

Create a leafmap `Map` centered on the randomly selected tile, add the randomly chosen XYZ tile service as a layer, and overlay the tile boundary polygon so you can see exactly which grid cell is being displayed.

In [ ]:
import leafmap
from ipyleaflet import ImageOverlay

# Build the concrete URL for the single selected tile
z, x, y = int(random_tile["z_val"]), int(random_tile["x_val"]), int(random_tile["y_val"])
selected_tile_url = service_url.format(z=z, x=x, y=y)
print(f"Selected tile URL: {selected_tile_url}")

# Get the geographic bounds of the selected tile for the image overlay
t_bounds = mercantile.bounds(mercantile.Tile(x, y, z))
image_bounds = [[t_bounds.south, t_bounds.west], [t_bounds.north, t_bounds.east]]

# Create a map centered on the random tile's centroid
m = leafmap.Map(center=[cy, cx], zoom=zoom)

# Add only the single tile image as an overlay (not the full tile service)
image_overlay = ImageOverlay(
    url=selected_tile_url,
    bounds=image_bounds,
    name=f"{service_name} ({z}/{x}/{y})"
)
m.add(image_overlay)

# Add the selected tile boundary outline
selected_tile_gdf = gpd.GeoDataFrame([random_tile], geometry="geometry", crs=grid_gdf.crs)
m.add_gdf(
    selected_tile_gdf,
    layer_name="Selected Tile Boundary",
    style={"color": "red", "weight": 2, "fillOpacity": 0.0}
)

m

## 9. Enhanced View: Load Child Tiles at the Next Zoom Level

In the Web Mercator tiling system, each tile at zoom level **z** is subdivided into **4 child tiles** at zoom level **z+1**. The children of tile `(z, x, y)` are:

| Child     | Coordinates         |
|-----------|---------------------|
| Top-left  | `(z+1, 2x, 2y)`    |
| Top-right | `(z+1, 2x+1, 2y)`  |
| Bottom-left  | `(z+1, 2x, 2y+1)`  |
| Bottom-right | `(z+1, 2x+1, 2y+1)`|

By loading all 4 child tiles we cover the same geographic area as the parent tile but at **2× the spatial resolution**. This means when you zoom in on the map, the imagery stays sharp.

In [ ]:
import leafmap
from ipyleaflet import ImageOverlay

# Calculate the 4 child tiles at zoom + 1
child_zoom = z + 1
child_tiles = [
    (child_zoom, 2 * x,     2 * y),      # top-left
    (child_zoom, 2 * x + 1, 2 * y),      # top-right
    (child_zoom, 2 * x,     2 * y + 1),  # bottom-left
    (child_zoom, 2 * x + 1, 2 * y + 1),  # bottom-right
]

print(f"Parent tile:  {z}/{x}/{y}  (zoom {z})")
print(f"Child tiles at zoom {child_zoom}:")
for cz, cx_t, cy_t in child_tiles:
    print(f"  {cz}/{cx_t}/{cy_t}")

# Create a map centered on the same tile centroid, starting at the parent zoom
m2 = leafmap.Map(center=[cy, cx], zoom=zoom)

# Add each child tile as an ImageOverlay
for cz, cx_t, cy_t in child_tiles:
    # Build the concrete URL for this child tile
    child_url = service_url.format(z=cz, x=cx_t, y=cy_t)

    # Get the geographic bounds of this child tile
    child_bounds = mercantile.bounds(mercantile.Tile(cx_t, cy_t, cz))
    child_image_bounds = [
        [child_bounds.south, child_bounds.west],
        [child_bounds.north, child_bounds.east]
    ]

    child_overlay = ImageOverlay(
        url=child_url,
        bounds=child_image_bounds,
        name=f"{service_name} ({cz}/{cx_t}/{cy_t})"
    )
    m2.add(child_overlay)

# Add the parent tile boundary outline for reference
selected_tile_gdf = gpd.GeoDataFrame([random_tile], geometry="geometry", crs=grid_gdf.crs)
m2.add_gdf(
    selected_tile_gdf,
    layer_name="Parent Tile Boundary",
    style={"color": "red", "weight": 2, "fillOpacity": 0.0}
)

print(f"\nNAIP Year: {naip_year} | Band Type: {band_type}")
print(f"Showing 4 child tiles from {service_name}")

m2